# Bike Shop Sales Analysis

## Objective
This project analyzes consumer spending data to identify key drivers of revenue, profitability, and customer purchasing behavior.

## Key Goals
- Identify top-performing product categories
- Evaluate profitability vs volume tradeoffs
- Analyze customer behavior by age group
- Provide actionable business recommendations

## Tools Used
- Python (pandas)
- SQL (DuckDB)
- Jupyter Notebook
- Tableau (for visualization)

In [1]:
import duckdb
import pandas as pd
import os
%load_ext sql
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [2]:
df = pd.read_csv("../Data/consumer_spending_cleaned.csv")
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34867 entries, 0 to 34866
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            34867 non-null  int64  
 1   orderDate        34866 non-null  object 
 2   customerAge      34866 non-null  float64
 3   customerGender   34866 non-null  object 
 4   country          34866 non-null  object 
 5   state            34866 non-null  object 
 6   productCategory  34866 non-null  object 
 7   subCategory      34866 non-null  object 
 8   quantity         34866 non-null  float64
 9   unitCost         34866 non-null  object 
 10  unitPrice        34866 non-null  object 
 11  totalCost        34866 non-null  object 
 12  totalRevenue     34867 non-null  object 
 13  totalProfit      34867 non-null  object 
 14  profitMargin     34867 non-null  float64
dtypes: float64(3), int64(1), object(11)
memory usage: 4.0+ MB


## Data Cleaning

The dataset required preprocessing to ensure accurate analysis:
- Converted currency fields to numeric values
- Handled missing subcategory values
- Standardized data types (dates, quantities)

This step ensures reliable financial and operational insights.

In [3]:
money_cols = ["unitCost", "unitPrice", "totalCost", "totalRevenue", "totalProfit"]

for col in money_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("(", "-", regex=False)
        .str.replace(")", "", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [4]:
df["orderDate"] = pd.to_datetime(df["orderDate"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["customerAge"] = pd.to_numeric(df["customerAge"], errors="coerce")

In [5]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34867 entries, 0 to 34866
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   index            34867 non-null  int64         
 1   orderDate        34866 non-null  datetime64[ns]
 2   customerAge      34866 non-null  float64       
 3   customerGender   34866 non-null  object        
 4   country          34866 non-null  object        
 5   state            34866 non-null  object        
 6   productCategory  34866 non-null  object        
 7   subCategory      34866 non-null  object        
 8   quantity         34866 non-null  float64       
 9   unitCost         34866 non-null  float64       
 10  unitPrice        34866 non-null  float64       
 11  totalCost        34866 non-null  float64       
 12  totalRevenue     34867 non-null  float64       
 13  totalProfit      34867 non-null  float64       
 14  profitMargin     34867 non-null  float

,index,orderDate,customerAge,customerGender,country,state,productCategory,subCategory,quantity,unitCost,unitPrice,totalCost,totalRevenue,totalProfit,profitMargin
0,203,2016-03-27,51.0,M,United States,Oregon,Accessories,Bike Racks,3.0,360.0,444.33,1080.0,1333.0,253.0,18.98
1,547,2016-03-31,43.0,F,United States,California,Accessories,Bike Racks,2.0,660.0,867.00,1320.0,1734.0,414.0,23.88
2,1631,2016-03-04,38.0,F,United States,California,Accessories,Bike Racks,2.0,240.0,290.50,480.0,581.0,101.0,17.38
3,1632,2016-05-30,38.0,F,United States,California,Accessories,Bike Racks,1.0,240.0,274.00,240.0,274.0,34.0,12.41
4,1633,2016-06-28,38.0,F,United States,California,Accessories,Bike Racks,3.0,400.0,505.00,1200.0,1515.0,315.0,20.79


In [6]:
%sql --persist df

Running query in 'duckdb:///:memory:'

Success! Persisted df to the database.

## Product Performance Analysis

### Business Question
Which product subcategories drive the most revenue, profit, and margin?

### Insight
- Helmets generate the highest total profit
- Bike racks have the highest profit margin
- Tires & Tubes have the highest sales volume

This reveals a clear tradeoff between demand (volume), profitability, and operational efficiency across product categories.

In [7]:
%%sql
SELECT 
    subCategory,
    SUM(quantity) AS total_quantity,
    SUM(totalRevenue) AS total_revenue,
    SUM(totalProfit) AS total_profit
FROM df
GROUP BY subCategory
ORDER BY total_profit DESC;

Running query in 'duckdb:///:memory:'

subCategory,total_quantity,total_revenue,total_profit
Helmets,8387.0,2738210.0,518475.0
Tires and Tubes,22213.0,2865915.0,512124.0
Jerseys,4033.0,1834110.0,300876.0
Mountain Bikes,5499.0,5176456.0,144627.0
Bottles and Cages,10558.0,709407.0,129567.0
Road Bikes,6119.0,3921989.0,98166.0
Touring Bikes,2673.0,2387910.0,94808.0
Shorts,1129.0,689184.0,87044.0
Hydration Packs,786.0,403276.0,72341.0
Fenders,1494.0,329204.0,71403.0


## Profitability vs Volume Tradeoff

High sales volume does not necessarily indicate high profitability.

- Tires & Tubes → high volume (traffic driver)
- Helmets → high profit (core revenue driver)
- Bike Racks → high margin (efficiency opportunity)

This suggests an opportunity to increase overall profitability by bundling high-margin products with high-volume items to maximize both revenue and efficiency.

In [8]:
%%sql
SELECT 
    subCategory,
    SUM(quantity) AS total_quantity,
    SUM(totalRevenue) AS total_revenue,
    SUM(totalProfit) AS total_profit,
    ROUND(SUM(totalProfit) / SUM(totalRevenue) * 100, 2) AS profit_margin_pct
FROM df
WHERE subCategory IS NOT NULL
  AND subCategory != 'None'
GROUP BY subCategory
ORDER BY total_profit DESC;

Running query in 'duckdb:///:memory:'

subCategory,total_quantity,total_revenue,total_profit,profit_margin_pct
Helmets,8387.0,2738210.0,518475.0,18.93
Tires and Tubes,22213.0,2865915.0,512124.0,17.87
Jerseys,4033.0,1834110.0,300876.0,16.4
Mountain Bikes,5499.0,5176456.0,144627.0,2.79
Bottles and Cages,10558.0,709407.0,129567.0,18.26
Road Bikes,6119.0,3921989.0,98166.0,2.5
Touring Bikes,2673.0,2387910.0,94808.0,3.97
Shorts,1129.0,689184.0,87044.0,12.63
Hydration Packs,786.0,403276.0,72341.0,17.94
Fenders,1494.0,329204.0,71403.0,21.69


In [9]:
%%sql
SELECT 
    subCategory,
    SUM(totalRevenue) AS total_revenue,
    SUM(totalProfit) AS total_profit,
    ROUND(SUM(totalProfit) / SUM(totalRevenue) * 100, 2) AS profit_margin_pct,
    RANK() OVER (ORDER BY SUM(totalProfit) DESC) AS profit_rank
FROM df
WHERE subCategory IS NOT NULL
  AND subCategory != 'None'
GROUP BY subCategory
ORDER BY profit_rank;


Running query in 'duckdb:///:memory:'

subCategory,total_revenue,total_profit,profit_margin_pct,profit_rank
Helmets,2738210.0,518475.0,18.93,1
Tires and Tubes,2865915.0,512124.0,17.87,2
Jerseys,1834110.0,300876.0,16.4,3
Mountain Bikes,5176456.0,144627.0,2.79,4
Bottles and Cages,709407.0,129567.0,18.26,5
Road Bikes,3921989.0,98166.0,2.5,6
Touring Bikes,2387910.0,94808.0,3.97,7
Shorts,689184.0,87044.0,12.63,8
Hydration Packs,403276.0,72341.0,17.94,9
Fenders,329204.0,71403.0,21.69,10


## Customer Segmentation Analysis

### Business Question
How do purchasing patterns differ across age groups?

### Insight
- Purchasing patterns are consistent across age groups
- The 25–44 age group drives the majority of demand
- Older customers show a slight shift toward safety products (helmets)

This identifies the core customer segment and highlights opportunities for targeted marketing and product positioning strategies.

In [10]:
%%sql
WITH ranked AS (
    SELECT 
        CASE 
            WHEN customerAge BETWEEN 15 AND 24 THEN '15-24'
            WHEN customerAge BETWEEN 25 AND 34 THEN '25-34'
            WHEN customerAge BETWEEN 35 AND 44 THEN '35-44'
            WHEN customerAge BETWEEN 45 AND 54 THEN '45-54'
            WHEN customerAge BETWEEN 55 AND 64 THEN '55-64'
            ELSE '65+'
        END AS age_group,
        subCategory,
        COUNT(*) AS purchase_count,
        ROW_NUMBER() OVER (
            PARTITION BY 
                CASE 
                    WHEN customerAge BETWEEN 15 AND 24 THEN '15-24'
                    WHEN customerAge BETWEEN 25 AND 34 THEN '25-34'
                    WHEN customerAge BETWEEN 35 AND 44 THEN '35-44'
                    WHEN customerAge BETWEEN 45 AND 54 THEN '45-54'
                    WHEN customerAge BETWEEN 55 AND 64 THEN '55-64'
                    ELSE '65+'
                END
            ORDER BY COUNT(*) DESC
        ) AS rank
    FROM df
    WHERE subCategory IS NOT NULL
    GROUP BY age_group, subCategory
)

SELECT *
FROM ranked
WHERE rank <= 3
ORDER BY age_group, rank;

Running query in 'duckdb:///:memory:'

age_group,subCategory,purchase_count,rank
15-24,Tires and Tubes,2073,1
15-24,Bottles and Cages,745,2
15-24,Helmets,584,3
25-34,Tires and Tubes,3287,1
25-34,Bottles and Cages,1740,2
25-34,Helmets,1409,3
35-44,Tires and Tubes,2950,1
35-44,Bottles and Cages,1554,2
35-44,Helmets,1215,3
45-54,Tires and Tubes,1893,1


In [11]:
%%sql
WITH ranked AS (
    SELECT 
        CASE 
            WHEN customerAge BETWEEN 15 AND 24 THEN '15-24'
            WHEN customerAge BETWEEN 25 AND 34 THEN '25-34'
            WHEN customerAge BETWEEN 35 AND 44 THEN '35-44'
            WHEN customerAge BETWEEN 45 AND 54 THEN '45-54'
            WHEN customerAge BETWEEN 55 AND 64 THEN '55-64'
            ELSE '65+'
        END AS age_group,
        subCategory,
        SUM(quantity) AS total_units,
        ROW_NUMBER() OVER (
            PARTITION BY 
                CASE 
                    WHEN customerAge BETWEEN 15 AND 24 THEN '15-24'
                    WHEN customerAge BETWEEN 25 AND 34 THEN '25-34'
                    WHEN customerAge BETWEEN 35 AND 44 THEN '35-44'
                    WHEN customerAge BETWEEN 45 AND 54 THEN '45-54'
                    WHEN customerAge BETWEEN 55 AND 64 THEN '55-64'
                    ELSE '65+'
                END
            ORDER BY SUM(quantity) DESC
        ) AS rank
    FROM df
    WHERE subCategory IS NOT NULL
    GROUP BY age_group, subCategory
)

SELECT *
FROM ranked
WHERE rank <= 3
ORDER BY age_group, rank;

Running query in 'duckdb:///:memory:'

age_group,subCategory,total_units,rank
15-24,Tires and Tubes,4112.0,1
15-24,Bottles and Cages,1490.0,2
15-24,Helmets,1147.0,3
25-34,Tires and Tubes,6574.0,1
25-34,Bottles and Cages,3455.0,2
25-34,Helmets,2842.0,3
35-44,Tires and Tubes,5853.0,1
35-44,Bottles and Cages,3106.0,2
35-44,Helmets,2460.0,3
45-54,Tires and Tubes,3863.0,1


## Business Recommendations

Based on the analysis:

- Promote high-margin products (bike racks) through bundling strategies to improve profitability
- Use high-volume products (tires & tubes) as traffic drivers for cross-selling opportunities
- Focus marketing efforts on the 25–44 age group, which represents the core customer base
- Emphasize safety-oriented products (helmets) when targeting older customers

These strategies can improve both revenue generation and profit margins.

## Limitations

- The dataset does not include customer income, purchase intent, or repeat purchase behavior
- Analysis assumes historical trends are indicative of future performance
- Some product categories may have limited representation

These limitations should be considered when interpreting the results.

In [12]:
# -----------------------------
# Create Tableau Datasets
# -----------------------------

# 1. Product Performance Dataset
product_df = duckdb.sql("""
SELECT 
    subCategory,
    SUM(quantity) AS total_units,
    SUM(totalRevenue) AS total_revenue,
    SUM(totalProfit) AS total_profit,
    ROUND(SUM(totalProfit) / SUM(totalRevenue) * 100, 2) AS profit_margin_pct
FROM df
WHERE subCategory IS NOT NULL
  AND subCategory != 'None'
GROUP BY subCategory
ORDER BY total_profit DESC
""").df()


# 2. Age Group Dataset
age_df = duckdb.sql("""
SELECT 
    CASE 
        WHEN customerAge BETWEEN 15 AND 24 THEN '15-24'
        WHEN customerAge BETWEEN 25 AND 34 THEN '25-34'
        WHEN customerAge BETWEEN 35 AND 44 THEN '35-44'
        WHEN customerAge BETWEEN 45 AND 54 THEN '45-54'
        WHEN customerAge BETWEEN 55 AND 64 THEN '55-64'
        ELSE '65+'
    END AS age_group,
    subCategory,
    SUM(quantity) AS total_units
FROM df
WHERE subCategory IS NOT NULL
  AND subCategory != 'None'
GROUP BY age_group, subCategory
""").df()


# 3. Time Trend Dataset
time_df = duckdb.sql("""
SELECT 
    DATE_TRUNC('month', orderDate) AS month,
    SUM(totalRevenue) AS total_revenue,
    SUM(totalProfit) AS total_profit
FROM df
GROUP BY month
ORDER BY month
""").df()


# -----------------------------
# Export to CSV (Tableau folder)
# -----------------------------

# product_df.to_csv("../Tableau/product_performance.csv", index=False)
# age_df.to_csv("../Tableau/age_analysis.csv", index=False)
# time_df.to_csv("../Tableau/time_trend.csv", index=False)

# print("CSV files successfully created in Tableau folder.")